In [3]:
import os
import time
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame
from pyspark.sql.functions import col
from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    read_latest_batch,
    write_clickhouse,
)
from etl.transformations.dimensions.scd.common import (
    add_hash,
    build_expired_version,
    build_new_version,
    split_changes,
)

MINIO_STAGING_BUCKET = os.environ.get("MINIO_STAGING_BUCKET", "staging")

In [5]:
spark = create_spark("load_dim_crop")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/21 17:11:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
user_role_df = read_latest_batch(
    spark,
    MINIO_STAGING_BUCKET,
    "user_roles",
)

user_df = read_latest_batch(
    spark,
    MINIO_STAGING_BUCKET,
    "users",
)

role_df = read_latest_batch(
    spark,
    MINIO_STAGING_BUCKET,
    "roles",
)

farm_df = read_latest_batch(
    spark,
    MINIO_STAGING_BUCKET,
    "farms",
)

user_role_df.show()
user_df.show()
role_df.show()
farm_df.show()

26/07/21 17:11:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---+-------+-------+-------+----------+----------+
| id|user_id|role_id|farm_id|created_at|updated_at|
+---+-------+-------+-------+----------+----------+
|  1|      1|      3|   NULL|1784637264|1784637264|
+---+-------+-------+-------+----------+----------+

+---+-----------------+--------------------+--------------------+---------+----------+----------+
| id|            email|       password_hash|           full_name|is_active|created_at|updated_at|
+---+-----------------+--------------------+--------------------+---------+----------+----------+
|  1|admin@example.com|pbkdf2_sha256$a88...|System Administrator|     true|1784637264|1784637264|
+---+-----------------+--------------------+--------------------+---------+----------+----------+

+---+---------------+--------------------+----------+----------+
| id|           name|         description|created_at|updated_at|
+---+---------------+--------------------+----------+----------+
|  1|   Farm Manager|Manager responsib...|1784637235|

In [7]:
dim_user_farm_role_df = (
    user_role_df
    .join(
        user_df,
        user_role_df.user_id == user_df.id,
        "left",
    )
    .join(
        role_df,
        user_role_df.role_id == role_df.id,
        "left",
    )
    .join(
        farm_df,
        user_role_df.farm_id == farm_df.id,
        "left",
    )
    .select(
        user_role_df.id.alias("user_role_id"),
        user_role_df.user_id,
        user_role_df.role_id,
        user_role_df.farm_id,
        user_df.full_name.alias("user_full_name"),
        role_df.name.alias("role_name"),
        farm_df.name.alias("farm_name"),
    )
)

dim_user_farm_role_df.show()

+------------+-------+-------+-------+--------------------+---------+---------+
|user_role_id|user_id|role_id|farm_id|      user_full_name|role_name|farm_name|
+------------+-------+-------+-------+--------------------+---------+---------+
|           1|      1|      3|   NULL|System Administrator|    Admin|     NULL|
+------------+-------+-------+-------+--------------------+---------+---------+



In [10]:
current_dim_df = (
            read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_user_farm_role FINAL
                ) AS dim_user_farm_role
                """,
        ).filter(col("is_current") == 1)
        )

current_dim_df.show()

+-------------+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+
|user_role_key|user_role_id|user_id|role_id|farm_key|farm_id|user_full_name|role_name|farm_name|valid_from|valid_to|is_current|_version|
+-------------+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+
+-------------+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+



In [12]:
source_hashed_df = add_hash(
    dim_user_farm_role_df,
    [
        "user_id",
        "role_id",
        "farm_id",
        "user_full_name",
        "role_name",
        "farm_name",
    ],
)

current_hashed_df = add_hash(
    current_dim_df,
    [
        "user_id",
        "role_id",
        "farm_id",
        "user_full_name",
        "role_name",
        "farm_name",
    ],
)

source_hashed_df.show()
current_hashed_df.show()

+------------+-------+-------+-------+--------------------+---------+---------+--------------------+
|user_role_id|user_id|role_id|farm_id|      user_full_name|role_name|farm_name|               _hash|
+------------+-------+-------+-------+--------------------+---------+---------+--------------------+
|           1|      1|      3|   NULL|System Administrator|    Admin|     NULL|b1a6955184386e5b1...|
+------------+-------+-------+-------+--------------------+---------+---------+--------------------+

+-------------+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+-----+
|user_role_key|user_role_id|user_id|role_id|farm_key|farm_id|user_full_name|role_name|farm_name|valid_from|valid_to|is_current|_version|_hash|
+-------------+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+-----+
+-------------+------------+-------+-------+--------+-------+----

In [14]:
new_rows_df, changed_rows_df = split_changes(
    source_hashed_df,
    current_hashed_df,
    "user_role_id",
)

load_version = int(time.time() * 1000)

build_new_version(
        new_rows_df,
        load_version,
    ).show()

build_expired_version(
            changed_rows_df,
            load_version,
            "user_role_key",
        ).show()

# Changed rows generate:
# 1. expired old version
# 2. new active version
rows_to_write = (
    build_new_version(
        new_rows_df,
        load_version,
    )
    .unionByName(
        build_expired_version(
            changed_rows_df,
            load_version,
            "user_role_key",
        )
    )
    .unionByName(
        build_new_version(
            changed_rows_df,
            load_version,
        )
    )
)

rows_to_write.show()

+------------+-------+-------+-------+--------------------+---------+---------+--------------------+-------------------+----------+-------------+
|user_role_id|user_id|role_id|farm_id|      user_full_name|role_name|farm_name|          valid_from|           valid_to|is_current|     _version|
+------------+-------+-------+-------+--------------------+---------+---------+--------------------+-------------------+----------+-------------+
|           1|      1|      3|   NULL|System Administrator|    Admin|     NULL|2026-07-21 17:29:...|2099-12-31 23:59:59|         1|1784654981043|
+------------+-------+-------+-------+--------------------+---------+---------+--------------------+-------------------+----------+-------------+

+------------+-------+-------+--------+-------+--------------+---------+---------+----------+--------+----------+--------+
|user_role_id|user_id|role_id|farm_key|farm_id|user_full_name|role_name|farm_name|valid_from|valid_to|is_current|_version|
+------------+-------+-

AnalysisException: [NUM_COLUMNS_MISMATCH] UNION can only be performed on inputs with the same number of columns, but the first input has 11 columns and the second input has 12 columns. SQLSTATE: 42826;
'Union false, false
:- Project [user_role_id#120L, user_id#1L, role_id#2L, farm_id#3, user_full_name#121, role_name#122, farm_name#123, valid_from#479, valid_to#480, is_current#481, 1784654981043 AS _version#482L]
:  +- Project [user_role_id#120L, user_id#1L, role_id#2L, farm_id#3, user_full_name#121, role_name#122, farm_name#123, valid_from#479, valid_to#480, 1 AS is_current#481]
:     +- Project [user_role_id#120L, user_id#1L, role_id#2L, farm_id#3, user_full_name#121, role_name#122, farm_name#123, valid_from#479, cast(2099-12-31 23:59:59 as timestamp) AS valid_to#480]
:        +- Project [user_role_id#120L, user_id#1L, role_id#2L, farm_id#3, user_full_name#121, role_name#122, farm_name#123, current_timestamp() AS valid_from#479]
:           +- Project [user_role_id#120L, user_id#1L, r

In [10]:
# write_clickhouse(
#     rows_to_write,
#     "dim_farm",
# )

26/07/21 15:43:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
